<!-- CELL 1 -->
# ARIMA GMV Forecast

Trains a per-partner auto-ARIMA model (weekly seasonal) on daily GMV and writes results to
`commercial_analytics.commercial_analytics.gmv_arima_forecast`.

Uses the same partner eligibility rule (>= 365 days history, active within last 60 days)
and the same current-month / next-month forecast window as the ai_forecast() job and the
Prophet notebook, so the dashboard can join all three model outputs side by side.

In [ ]:
# CELL 2
# -----------------------------------------------------------------------------
# STEP 1: INSTALL DEPENDENCIES
# -----------------------------------------------------------------------------
# pmdarima provides auto_arima(), which automatically searches for the best
# (p,d,q)(P,D,Q) order instead of us having to hand-pick ARIMA parameters per partner.
# This only needs to run once per cluster session -- if the cluster is reused across
# days without restarting, this cell can be skipped on subsequent runs.
%pip install pmdarima

In [ ]:
# CELL 3
# -----------------------------------------------------------------------------
# STEP 2: RESTART PYTHON
# -----------------------------------------------------------------------------
# Databricks requires a Python kernel restart after %pip install so the newly
# installed package is actually importable. Must run immediately after Step 1,
# before any cell that imports pmdarima or references variables below --
# restarting clears all previously defined variables in this session.
dbutils.library.restartPython()

In [ ]:
# CELL 4
# -----------------------------------------------------------------------------
# STEP 3: LOAD HISTORICAL DAILY GMV
# -----------------------------------------------------------------------------
# Pull ~4 years of daily GMV per partner from the same source table used by the
# ai_forecast() SQL job and the Prophet notebook, so all three models are trained
# on identical underlying data.
#   ds      -- calendar date of the loan/transaction
#   partner -- partner_grouping_legacy, the grain we forecast at
#   y       -- total GMV booked that day for that partner
import pandas as pd
import numpy as np

df = spark.sql("""
    SELECT
        DATE(loan_date_etz)     AS ds,
        partner_grouping_legacy AS partner,
        SUM(gmv_amount_usd)     AS y
    FROM commercial_analytics.public.forecasting
    WHERE
        loan_date_etz      >= '2022-01-01'
        AND loan_date_etz   <  CURRENT_DATE
        AND gmv_amount_usd IS NOT NULL
        AND gmv_amount_usd  >= 0
        AND partner_grouping_legacy IS NOT NULL
    GROUP BY 1, 2
    ORDER BY 1, 2
""").toPandas()

# Prophet/ARIMA libraries expect a real datetime dtype, not a plain date object
df["ds"] = pd.to_datetime(df["ds"])

print(f"Loaded {df['partner'].nunique()} partners, {len(df)} rows")
print(f"Date range: {df['ds'].min().date()} to {df['ds'].max().date()}")

In [ ]:
# CELL 5
# -----------------------------------------------------------------------------
# STEP 4: SPLIT PARTNERS INTO ARIMA-ELIGIBLE VS. FALLBACK
# -----------------------------------------------------------------------------
# Same eligibility rule used by ai_forecast() and the Prophet notebook, so all
# three models agree on which partners get a real statistical forecast vs. a
# simple average:
#   - >= 365 days of history     -> enough data for ARIMA to learn seasonality/trend
#   - active in the last 60 days -> partner is still generating GMV, not dormant
today     = pd.Timestamp.today().normalize()
cutoff_60 = today - pd.Timedelta(days=60)

# One row per partner: how many days of data they have, and their most recent date
partner_stats = (
    df.groupby("partner")
    .agg(days=("ds", "count"), last_date=("ds", "max"))
    .reset_index()
)

# Partners that pass both checks -> ARIMA
eligible = partner_stats[
    (partner_stats["days"] >= 365) &
    (partner_stats["last_date"] >= cutoff_60)
]["partner"].tolist()

# Everyone else -> weighted moving average fallback (Step 6)
ineligible = partner_stats[
    ~partner_stats["partner"].isin(eligible)
]["partner"].tolist()

print(f"Eligible for ARIMA:    {len(eligible)} partners")
print(f"Ineligible (fallback): {len(ineligible)} partners")

In [ ]:
# CELL 6
# -----------------------------------------------------------------------------
# STEP 5: TRAIN AUTO-ARIMA PER ELIGIBLE PARTNER
# -----------------------------------------------------------------------------
# auto_arima searches over ARIMA orders and picks the best fit per partner --
# no manual (p,d,q) tuning needed. seasonal=True, m=7 adds a weekly seasonal
# component (SARIMA) since daily GMV typically has day-of-week patterns
# (e.g. weekends lower than weekdays), similar to Prophet's weekly_seasonality.
#
# Speed tuning: trains on the trailing 2 years only (not the full history),
# bounds the search space, and fixes D=1 instead of auto-detecting it --
# all to cut per-partner fit time down without materially hurting fit quality
# for a weekly-only seasonal pattern.
from pmdarima import auto_arima
import warnings
import time
warnings.filterwarnings("ignore")

# Forecast window: from today through the end of NEXT month.
# This matches the ai_forecast()/Prophet horizon so all models cover the same dates.
current_month_start = today.replace(day=1)
next_month_start     = current_month_start + pd.DateOffset(months=1)
next_month_end       = next_month_start + pd.offsets.MonthEnd(1)
n_periods             = (next_month_end - today).days + 1

results = []
failed  = []

total_partners = len(eligible)
loop_start = time.time()

for i, partner in enumerate(eligible, 1):
    partner_start = time.time()
    try:
        # Isolate this partner's series and index it by date
        partner_df = df[df["partner"] == partner][["ds", "y"]].copy()
        partner_df = partner_df.set_index("ds").asfreq("D")
        # asfreq() introduces NaN for any date with no transactions -- ARIMA requires
        # a gap-free series, so treat missing days as $0 GMV
        partner_df["y"] = partner_df["y"].fillna(0)

        # Only use the trailing 2 years for fitting -- enough for weekly seasonality
        # and recent trend, much faster than fitting on the full ~4-year history
        partner_df = partner_df[partner_df.index >= today - pd.Timedelta(days=730)]

        # Fit the model: weekly seasonal SARIMA, auto-selected orders within bounds
        model = auto_arima(
            partner_df["y"],
            seasonal=True,
            m=7,
            D=1,              # assume one round of weekly differencing -- skips a slow auto-test
            max_p=3,
            max_q=3,
            max_P=1,
            max_Q=1,
            stepwise=True,
            suppress_warnings=True,
            error_action="ignore"
        )

        # Forecast n_periods days forward, with an 80% confidence interval
        # (alpha=0.20 -> 80% CI, matching the Prophet notebook's interval_width=0.80)
        forecast, conf_int = model.predict(
            n_periods=n_periods, return_conf_int=True, alpha=0.20
        )

        forecast_dates = pd.date_range(start=today, periods=n_periods, freq="D")

        results.append(pd.DataFrame({
            "ds":                       forecast_dates,
            "partner_grouping_legacy":  partner,
            "yhat":                     forecast,
            "yhat_lower":               conf_int[:, 0],
            "yhat_upper":               conf_int[:, 1],
        }))

        elapsed = time.time() - partner_start
        print(f"[{i}/{total_partners}] Trained: {partner} ({elapsed:.1f}s)")

    except Exception as e:
        elapsed = time.time() - partner_start
        print(f"[{i}/{total_partners}] Failed: {partner} - {e} ({elapsed:.1f}s)")
        failed.append(partner)

total_elapsed = time.time() - loop_start
print(f"\nCompleted: {len(results)} partners")
print(f"Failed:    {len(failed)} partners")
print(f"Total time: {total_elapsed:.1f}s ({total_elapsed/60:.1f} min)")
if failed:
    print(failed)

In [ ]:
# CELL 7
# -----------------------------------------------------------------------------
# STEP 6: FALLBACK FORECAST FOR INELIGIBLE PARTNERS
# -----------------------------------------------------------------------------
# Partners with < 365 days of history or no activity in the last 60 days do not
# have enough signal for ARIMA to fit reliably. Instead, use a simple
# recency-weighted moving average -- same method used by the SQL job's fallback
# (Section 3) and the Prophet notebook's fallback path.
fallback_results = []
# One forecast row per calendar day across current + next month, same as the ARIMA path
forecast_dates = pd.date_range(start=current_month_start, end=next_month_end, freq="D")

for partner in ineligible:
    partner_df = df[df["partner"] == partner][["ds", "y"]].copy()
    partner_df = partner_df.sort_values("ds")

    # Only look at the trailing year of data for the weighted average
    recent = partner_df[partner_df["ds"] >= today - pd.Timedelta(days=365)].copy()
    if len(recent) == 0:
        continue

    # Rank rows so the most recent day gets the highest weight (rank = len(recent)),
    # oldest day gets the lowest weight (rank = 1) -- a linearly recency-weighted average
    recent["rank"] = range(len(recent), 0, -1)
    wma = (recent["y"] * recent["rank"]).sum() / recent["rank"].sum()

    # Same flat daily forecast value for every day in the window, with a flat
    # +/- 15% confidence band (no day-of-week adjustment here, unlike the SQL job's
    # fallback which applies a dow_multiplier -- this is a simpler version)
    for fdate in forecast_dates:
        fallback_results.append({
            "ds":                       fdate,
            "partner_grouping_legacy":  partner,
            "yhat":                     max(round(wma, 2), 0),
            "yhat_lower":               max(round(wma * 0.85, 2), 0),
            "yhat_upper":               max(round(wma * 1.15, 2), 0),
        })

fallback_df = pd.DataFrame(fallback_results)
print(f"Fallback rows generated: {len(fallback_df)}")

In [ ]:
# CELL 8
# -----------------------------------------------------------------------------
# STEP 7: COMBINE + WRITE TO THE FORECAST TABLE
# -----------------------------------------------------------------------------
# Merge the ARIMA results (Step 5) and fallback results (Step 6) into one table,
# trim to exactly the current+next month window, label each row, and overwrite
# the Delta table so the dashboard always reads a fully fresh forecast.
all_results = pd.concat(
    results + ([fallback_df] if len(fallback_df) > 0 else []),
    ignore_index=True
)

all_results["ds"] = pd.to_datetime(all_results["ds"])

# ARIMA forecasts start at "today", which may fall mid-month, and could in theory
# extend slightly outside the target window -- clip to current-month-start through
# next-month-end to match the dashboard's expected range
all_results = all_results[
    (all_results["ds"] >= current_month_start) &
    (all_results["ds"] <= next_month_end)
].copy()

# Label each date as current_month or next_month, same convention used by
# gmv_forecast_output and gmv_prophet_forecast, so the dashboard join logic
# works identically across all three model tables
all_results["forecast_horizon"] = all_results["ds"].apply(
    lambda d: "current_month" if d < next_month_start else "next_month"
)

# Floor all forecast values at 0 -- GMV can't be negative, but ARIMA math can
# produce negative point estimates or lower bounds for low-volume partners
all_results["yhat"]       = all_results["yhat"].clip(lower=0).round(2)
all_results["yhat_lower"] = all_results["yhat_lower"].clip(lower=0).round(2)
all_results["yhat_upper"] = all_results["yhat_upper"].clip(lower=0).round(2)

all_results["run_date"] = today

# Tag which method produced each partner's row -- lets the dashboard/QA distinguish
# a real ARIMA forecast from the simple moving-average fallback
all_results["model"] = all_results["partner_grouping_legacy"].apply(
    lambda p: "arima" if p in eligible else "fallback_wma"
)

# Full overwrite each run (not an incremental append) -- keeps the table simple
# and guarantees no stale/duplicate rows from previous runs
spark.createDataFrame(all_results).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("commercial_analytics.commercial_analytics.gmv_arima_forecast")

print(f"Written {len(all_results)} rows for {all_results['partner_grouping_legacy'].nunique()} partners")

In [ ]:
# CELL 9
# -----------------------------------------------------------------------------
# STEP 8: SANITY CHECK
# -----------------------------------------------------------------------------
# Quick post-write validation: confirms partner counts, row counts, and total
# forecasted GMV look reasonable, broken out by horizon and by which model
# (arima vs. fallback_wma) produced the numbers. Useful as a first check if the
# dashboard numbers ever look off after this notebook runs.
validation = spark.sql("""
    SELECT
        forecast_horizon,
        model,
        COUNT(DISTINCT partner_grouping_legacy) AS partners,
        COUNT(*)                                AS forecast_days,
        ROUND(SUM(yhat), 2)                     AS total_forecast_gmv,
        MIN(ds)                                 AS first_date,
        MAX(ds)                                 AS last_date
    FROM commercial_analytics.commercial_analytics.gmv_arima_forecast
    GROUP BY forecast_horizon, model
    ORDER BY forecast_horizon, model
""")
display(validation)